[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/38_causal_machine_learning/01_correlation_vs_causation_dags.ipynb)

# Bài 01 — Tương quan, nhân quả và đồ thị nhân quả (DAG)

**🎯 Mục tiêu**
- Phân biệt câu hỏi **dự đoán** ("ai sẽ mua?") với câu hỏi **nhân quả** ("nếu phát coupon thì có mua thêm không?").
- **DAG**: ba cấu trúc cơ bản — chuỗi (mediator), nhánh (confounder), va chạm (collider) — và hệ quả khi kiểm soát (adjust) từng loại.
- Mô phỏng các nghịch lý kinh điển: **Simpson**, **Berkson** (collider bias), kiểm soát **mediator**.
- **Tiêu chí cửa sau** (backdoor criterion): kiểm soát biến nào thì ước lượng được tác động nhân quả.
- Vì sao **thí nghiệm ngẫu nhiên** là chuẩn vàng, và các giả định khi chỉ có dữ liệu quan sát.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/38_causal_machine_learning

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)
N = 20000


def coef(y, **features):
    """Hệ số hồi quy tuyến tính của y theo các đặc trưng (theo thứ tự truyền vào)."""
    X = np.column_stack(list(features.values()))
    return dict(zip(features, LinearRegression().fit(X, y).coef_.round(3)))

## 1. Dự đoán ≠ nhân quả

Ở các thành phố ven biển, số kem bán ra và số vụ đuối nước tương quan mạnh. Một mô hình dùng doanh số kem **dự đoán** đuối nước rất tốt — nhưng cấm bán kem không cứu được ai: cả hai đều do **nhiệt độ**.

In [ ]:
temperature = rng.normal(28, 5, N)
ice_cream = 50 + 10 * temperature + rng.normal(0, 20, N)
drowning = 0.2 * temperature + rng.normal(0, 1, N)                     # KHÔNG phụ thuộc kem
print(f"tương quan(kem, đuối nước) = {np.corrcoef(ice_cream, drowning)[0, 1]:.2f}")
print("hồi quy đuối nước ~ kem              :", coef(drowning, kem=ice_cream))
print("hồi quy đuối nước ~ kem + nhiệt độ   :", coef(drowning, kem=ice_cream, nhiet_do=temperature))

Kem là đặc trưng **dự đoán** tốt (khi không có nhiệt độ), nhưng hệ số của nó **không** phải tác động nhân quả. Khi kiểm soát nhiệt độ — nguyên nhân chung — hệ số của kem về 0.

ML truyền thống tối ưu **dự đoán** $\mathbb E[Y\mid X]$ dưới phân phối dữ liệu hiện có. Câu hỏi nhân quả hỏi điều gì xảy ra khi ta **can thiệp** thay đổi $X$: $\mathbb E[Y\mid do(X = x)]$ — phân phối dữ liệu sẽ khác đi. Mọi bài toán "nên làm gì" (phát coupon, kê thuốc, đổi giá, thay đổi chính sách) là câu hỏi nhân quả.

## 2. DAG: ba khối xây dựng

**Đồ thị nhân quả** (DAG — directed acyclic graph): mỗi mũi tên $A \to B$ nghĩa là $A$ là nguyên nhân trực tiếp của $B$. Mọi DAG ghép từ ba cấu trúc:

| Cấu trúc | Hình | Ví dụ | Không kiểm soát $Z$ | Kiểm soát $Z$ |
|---|---|---|---|---|
| Nhánh — **confounder** | $X \leftarrow Z \rightarrow Y$ | nhiệt độ → kem, đuối nước | $X$, $Y$ tương quan giả | **chặn** đường giả ✅ |
| Chuỗi — **mediator** | $X \rightarrow Z \rightarrow Y$ | đào tạo → kỹ năng → lương | thấy toàn bộ tác động | **chặn** cả tác động thật ❌ |
| Va chạm — **collider** | $X \rightarrow Z \leftarrow Y$ | tài năng → nổi tiếng ← ngoại hình | $X$, $Y$ độc lập | **mở** tương quan giả ❌ |

### 2.1. Nghịch lý Simpson (confounder)

Hai phương pháp điều trị; bệnh nhân **nặng** thường được chọn phương pháp A (mạnh hơn) và vốn có tỉ lệ khỏi thấp hơn:

In [ ]:
severe = rng.random(N) < 0.5
treat_A = rng.random(N) < np.where(severe, 0.8, 0.2)                   # người nặng thường được chọn A
recover_prob = np.where(severe, 0.30, 0.80) + 0.10 * treat_A           # A tốt hơn B 10 điểm % ở MỌI nhóm
recovered = rng.random(N) < recover_prob
df = pd.DataFrame({"nặng": severe, "điều trị A": treat_A, "khỏi": recovered})
overall = df.groupby("điều trị A")["khỏi"].mean()
by_group = df.groupby(["nặng", "điều trị A"])["khỏi"].mean().unstack()
print(f"GỘP CHUNG: tỉ lệ khỏi — B = {overall[False]:.1%}, A = {overall[True]:.1%} → A có vẻ TỆ hơn")
print("THEO NHÓM MỨC ĐỘ:\n", by_group.rename(columns={False: "B", True: "A"}).round(3))

Gộp chung, A trông **tệ hơn**; trong **từng** nhóm, A **tốt hơn** 10 điểm %. Không có gì sai trong số liệu — mức độ bệnh là **confounder** (ảnh hưởng cả việc chọn điều trị lẫn khả năng khỏi). Câu trả lời đúng cho "A có tốt hơn không?" là con số **theo nhóm** (kiểm soát confounder). Nhưng không phải lúc nào chia nhóm cũng đúng — xem 2.2 và 2.3; chỉ **cấu trúc nhân quả** mới cho biết nên kiểm soát gì.

### 2.2. Collider: nghịch lý Berkson

Trong dân số, tài năng và ngoại hình **độc lập**. Người trở nên **nổi tiếng** khi tổng tài năng + ngoại hình vượt ngưỡng. Chỉ nhìn **người nổi tiếng**:

In [ ]:
talent, looks = rng.normal(size=N), rng.normal(size=N)
famous = talent + looks > 2.0                                          # collider: cả hai cùng gây ra "nổi tiếng"
print(f"tương quan(tài năng, ngoại hình) — toàn dân số: {np.corrcoef(talent, looks)[0, 1]:+.2f} | "
      f"chỉ người nổi tiếng: {np.corrcoef(talent[famous], looks[famous])[0, 1]:+.2f}")
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(talent[~famous][:2000], looks[~famous][:2000], s=3, alpha=0.3, label="không nổi tiếng")
ax.scatter(talent[famous], looks[famous], s=3, alpha=0.6, label="nổi tiếng")
ax.set_xlabel("tài năng")
ax.set_ylabel("ngoại hình")
ax.legend(markerscale=4)
plt.tight_layout()
plt.show()

Chọn mẫu theo collider (chỉ nghiên cứu người nổi tiếng, chỉ bệnh nhân nhập viện, chỉ khách hàng còn hoạt động, chỉ hồ sơ **được duyệt**...) tạo ra tương quan **âm giả**: "người nổi tiếng đẹp thì thường kém tài". Đây là thiên lệch chọn mẫu rất phổ biến trong dữ liệu kinh doanh — ví dụ mô hình tín dụng chỉ train trên khoản vay **đã được duyệt** (chương 19, 36).

### 2.3. Mediator: kiểm soát làm mất tác động thật

Chương trình đào tạo tăng lương **thông qua** kỹ năng:

In [ ]:
training = (rng.random(N) < 0.5).astype(float)                         # phân ngẫu nhiên
skill = 2.0 * training + rng.normal(size=N)
salary = 10 + 3.0 * skill + rng.normal(size=N)                         # tác động toàn phần của đào tạo = 2 × 3 = 6
print("lương ~ đào tạo           :", coef(salary, dao_tao=training))
print("lương ~ đào tạo + kỹ năng :", coef(salary, dao_tao=training, ky_nang=skill))

Tác động **toàn phần** của đào tạo là 6; kiểm soát kỹ năng (mediator) cho hệ số ≈ 0 — kết luận sai "đào tạo vô ích". Kiểm soát mediator chỉ đúng nếu câu hỏi là tác động **trực tiếp** (không qua kỹ năng). Bài học: **"cứ thêm mọi biến vào hồi quy cho chắc"** là sai — thêm mediator hay collider đều làm hỏng ước lượng ("bad controls").

## 3. Tiêu chí cửa sau (backdoor criterion)

Để ước lượng tác động của $X$ lên $Y$ từ dữ liệu quan sát, chọn tập $Z$ sao cho:
1. $Z$ **chặn mọi đường cửa sau** từ $X$ tới $Y$ (đường bắt đầu bằng mũi tên **vào** $X$, như $X \leftarrow \text{confounder} \rightarrow Y$);
2. $Z$ **không** chứa hậu duệ của $X$ (mediator, hệ quả của điều trị, collider do $X$ gây ra).

Khi đó $\mathbb E[Y\mid do(X = x)] = \sum_z \mathbb E[Y\mid X = x, Z = z]\,P(Z = z)$ (công thức điều chỉnh). Ví dụ tổng hợp: DAG gồm confounder $C$, mediator $M$, collider $K$:

In [ ]:
C = rng.normal(size=N)                                                 # confounder
X_t = 0.8 * C + rng.normal(size=N)                                     # điều trị (liên tục)
M = 1.0 * X_t + rng.normal(size=N)                                     # mediator
Y_o = 0.5 * X_t + 1.0 * M + 1.5 * C + rng.normal(size=N)               # tác động toàn phần của X = 0.5 + 1.0 = 1.5
K = X_t + Y_o + rng.normal(size=N)                                     # collider
for name, ctrl in [("không kiểm soát gì", {}), ("kiểm soát C (đúng)", {"C": C}), ("C + M (sai: mediator)", {"C": C, "M": M}),
                   ("C + K (sai: collider)", {"C": C, "K": K})]:
    print(f"{name:24s}: hệ số của X = {coef(Y_o, X=X_t, **ctrl)['X']:+.3f}   (tác động nhân quả thật = 1.500)")

Chỉ đúng tập điều chỉnh do **DAG** chỉ định mới cho đúng 1.5. DAG không suy ra được từ dữ liệu một cách chắc chắn — nó là **giả định** dựa trên hiểu biết lĩnh vực, cần viết ra tường minh và tranh luận. Thư viện DoWhy (Microsoft) giúp mô hình hoá DAG, tìm tập điều chỉnh và kiểm tra độ nhạy.

## 4. Thí nghiệm ngẫu nhiên: chuẩn vàng

Nếu **phân ngẫu nhiên** điều trị (A/B test, thử nghiệm lâm sàng), điều trị độc lập với mọi confounder — kể cả những thứ không đo được → mọi đường cửa sau bị cắt → so sánh đơn giản hai nhóm là ước lượng không chệch. Khi không thể thí nghiệm (đắt, phi đạo đức, không khả thi), dùng dữ liệu quan sát với các **giả định** (bài 02–03):
- **Không có confounder chưa đo** (unconfoundedness) — không kiểm chứng được bằng dữ liệu;
- **Overlap/positivity**: mỗi loại cá thể đều có khả năng nhận cả hai điều trị;
- **SUTVA**: điều trị của người này không ảnh hưởng người khác; điều trị được định nghĩa rõ.

## 🧠 Kiểm tra nhanh

1. Vì sao một đặc trưng dự đoán tốt chưa chắc là nguyên nhân?
2. Confounder, mediator, collider là gì? Kiểm soát mỗi loại gây ra điều gì?
3. Giải thích nghịch lý Simpson bằng DAG.
4. Vì sao nghiên cứu chỉ trên người nổi tiếng/bệnh nhân nhập viện có thể tạo tương quan giả?
5. Phát biểu tiêu chí cửa sau.
6. Vì sao thí nghiệm ngẫu nhiên loại bỏ confounding? Ba giả định khi dùng dữ liệu quan sát?

## 📌 Tóm tắt

- Dự đoán: $\mathbb E[Y\mid X]$; nhân quả: $\mathbb E[Y\mid do(X)]$ — khác nhau khi có confounder.
- DAG: kiểm soát confounder (chặn cửa sau) — đúng; kiểm soát mediator hoặc collider — sai.
- Simpson, Berkson là hệ quả của cấu trúc nhân quả, không phải lỗi số liệu; "thêm mọi biến" không an toàn.
- RCT là chuẩn vàng; dữ liệu quan sát cần giả định không có confounder chưa đo, overlap, SUTVA.

## 📚 Tài liệu tham khảo

- Pearl & Mackenzie (2018) — *The Book of Why*
- Hernán & Robins (2020) — *Causal Inference: What If* (sách mở)
- Cunningham (2021) — *Causal Inference: The Mixtape* (sách mở)
- Cinelli, Forney & Pearl (2022) — *A Crash Course in Good and Bad Controls*
- [DoWhy](https://www.pywhy.org/dowhy/) — thư viện suy luận nhân quả